In [ ]:
# Download MovieLens ml-latest-small dataset directly
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip

In [2]:
from zipfile import ZipFile

# Path to downloaded zip
zip_path = "/content/ml-latest-small.zip"

# Extract all files
with ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/")

print("Extraction done!")

Extraction done!


In [ ]:

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
movies = pd.read_csv("/content/ml-latest-small/movies.csv")

# Fill missing values
movies["genres"] = movies["genres"].fillna("")

# Combine title + genres
movies["content"] = movies["title"] + " " + movies["genres"]

# Convert text into numbers
vectorizer = CountVectorizer(stop_words="english")
content_matrix = vectorizer.fit_transform(movies["content"])

# Calculate similarity
similarity = cosine_similarity(content_matrix)

# Recommendation function
def recommend(movie_name):

    movie_index = movies[movies["title"] == movie_name].index[0]

    distances = similarity[movie_index]

    pairs = list(enumerate(distances))

    pairs.sort(key=lambda x: x[1], reverse=True)

    recommendations = pairs[1:6]

    print("\nMovies similar to", movie_name, ":\n")

    for item in recommendations:
        index = item[0]
        print(movies.iloc[index].title, "-", movies.iloc[index].genres)


# Search function (for user input)
def recommend_from_search():

    name = input("Enter part of movie name: ")

    # Find all results
    results = movies[movies['title'].str.contains(name, case=False)]

    if results.empty:
        print("No movie found")
        return

    # Display results with numbers
    print("\nMovies found:\n")
    for i, (index, row) in enumerate(results.head(5).iterrows()): # Display up to 5 options
        print(f"{i+1}. {row['title']}")

    while True:
        try:
            choice = int(input(f"\nEnter the number of the movie you want to select (1-{min(5, len(results))}): "))
            if 1 <= choice <= min(5, len(results)): # Validate choice within displayed range
                # Get the actual title from the results DataFrame using the index
                movie_name_to_recommend = results.head(5).iloc[choice - 1]['title']
                break
            else:
                print(f"Invalid choice. Please enter a number between 1 and {min(5, len(results))}.")
        except ValueError:
            print("Invalid input. Please enter a number.")

    recommend(movie_name_to_recommend)


# Run the system
recommend_from_search()
